#3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
from os.path import join
from tqdm import tqdm


In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

In [3]:
drenageo = gpd.read_file(
    join(
        'data',
        'drenagem.zip'
    )
)
print(
    f'Shape: {drenageo.shape};\n'+
    f'\nSample:{drenageo.sample()}'
)

## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)

Shape: (27611, 15);

Sample:      cd_identif cd_tipo_ac tx_tipo_ac cd_numero_ nm_bairro nm_acident  \
5890      5537.0        RIO        RIO          3       S/B  RIO TIETE   

       qt_comprim  cd_tipo_cu                nm_tipo_cu            nm_via_pro  \
5890  2847.307983        11.0  Trecho em estado natural  TRES (FAZENDA ITAIM)   

     nm_descrit           nm_tipo_tr dt_atualiz cd_usuario  \
5890  Rio Tietê  Trecho a céu aberto 2025-01-03       None   

                                               geometry  
5890  LINESTRING (424454.937 7392424.356, 424453.07 ...  


0            1
1        26125
2            2
3        26126
4        26127
         ...  
27606    27596
27607    27597
27608    27598
27609    27599
27610    27600
Name: cd_identif, Length: 27611, dtype: int64

In [4]:
gdf= drenageo[[
    'cd_identif', 
    'cd_tipo_ac', 
    'cd_tipo_cu', 
    'nm_acident', 
    'geometry'
]]

Pelo que eu vi e conversei com o Mauryas: ```Considere os tipos para efeito de continuidade. Só não gere a nascente se esses tipos desconhecidos forem os finais de um curso.```  
Sabe o que isso quer dizer também? Que não dá pra eu separar por nomes igual o Henrique fez...

In [5]:
drenageo.sample(3) # o melhor era aqui ser um drenageo_in_estim, né, mas ok, vamos trabalhar só com o drenageo por enquanto

,cd_identif,cd_tipo_ac,tx_tipo_ac,cd_numero_,nm_bairro,nm_acident,qt_comprim,cd_tipo_cu,nm_tipo_cu,nm_via_pro,nm_descrit,nm_tipo_tr,dt_atualiz,cd_usuario,geometry
25809,24646.0,ND,None,4,None,SD,167.263651,11.0,Trecho em estado natural,None,None,Trecho a céu aberto,2025-01-03,None,"LINESTRING (332042.157 7353123.276, 332037.457..."
6634,6262.0,COR,CORREGO,5,None,CORREGO MANDAQUI,284.652283,9.0,Trecho canalizado a céu aberto,None,Córrego Mandaqui,Trecho a céu aberto,2025-01-03,None,"LINESTRING (330273.86 7400563.278, 330270.559 ..."
20993,20003.0,ND,None,1,RESIDENCIAL SOL NASCENTE,SD,390.823433,11.0,Trecho em estado natural,S JUDAS TADEU,None,Trecho a céu aberto,2025-01-03,None,"LINESTRING (318535.707 7407534.983, 318567.465..."


Eu descobri pq o touches não dava certo e agora acho que tudo isso foi meio em vão actually... faz pensar, né?  
Esse tempo todo, eu precisava usar o `.intersec()`, veja:

In [6]:
gdf.head(1)

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry
0,1.0,ND,11.0,SD,"LINESTRING (326392.101 7349046.69, 326389.141 ..."


In [7]:
pontos = gdf.copy()
pontos_buff = gdf.copy()
pontos['geometry']=shapely.get_point(gdf.geometry, 0)

Agora damos buffer nos pontos, para tirarmos os que intersectam com alguma coisa:

In [8]:
#vamos começar com um buffer de 10 metros e ir diminuindo aos poucos

pontos_buff['geometry'] = pontos['geometry'].buffer(10)

# Visualizar
### Ok, eu estava errada... mesmo com o intersec e um mega buffer nos pontos, ainda não dá certo, vamos voltar pra tatica do Henrique mesmo
m=drenageo.explore(color='pink')
pontos_buff.explore(
    m=m
)